# My First AI — GPU Training

Run this notebook in Google Colab with a CUDA runtime. It uses the repository's real PyTorch v5 BPE training code.

In [ ]:
REPO_URL = input('Paste your GitHub repository URL: ').strip()
!git clone $REPO_URL /content/my-first-ai-training
%cd /content/my-first-ai-training
!pip install -q -r training/requirements.txt
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import files
from pathlib import Path
Path('data').mkdir(exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    Path('data/train.txt').write_bytes(content)
print('Training corpus:', Path('data/train.txt').stat().st_size, 'bytes')

In [ ]:
!python training/prepare_data.py --input data/train.txt --out data/processed --vocab-size 1024

In [ ]:
!python training/train.py --data-dir data/processed --steps 1000 --batch-size 32 --lr 0.0003 --seed 42 --save-every 500 --eval-every 100 --out checkpoints/my-first-ai-v5-bpe.pt

In [ ]:
!python training/evaluate.py --checkpoint checkpoints/my-first-ai-v5-bpe.pt --data-dir data/processed

## Resume training

To continue from the saved checkpoint, run the trainer again with `--resume` and set `--steps` to the number of additional optimizer steps.

In [ ]:
# Example:
# !python training/train.py --data-dir data/processed --resume checkpoints/my-first-ai-v5-bpe.pt --steps 1000 --batch-size 32 --lr 0.0003 --seed 42 --out checkpoints/my-first-ai-v5-bpe.pt